In [ ]:
import pandas as pd
import numpy as np

column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
}

In [ ]:
data_gdynia = pd.read_csv(
    "../data/raw/srodmiescie_raw_object_id.csv", sep=";", index_col=0, dtype=column_types
).dropna(subset="ground_floor_function")

data_gdynia.head(20)

In [ ]:
data_extract = (
    data_gdynia.assign(
        funkcja_parter=lambda d: (
            d["ground_floor_function"]
            .str.replace("_", " ")
            .str.lower()
        ))
)

Exercise 1: Number of groups

In [ ]:
num_streets = data_extract.groupby('addr_street').ngroups
print(f"number of individual streets: {num_streets}")
data_extract.groupby('addr_street').get_group('Stefana Batorego').head()

In [ ]:
assert num_streets == data_extract["addr_street"].nunique()

Exercise 2:Number of buildings per street in 2 methods

In [ ]:
num_count = data_extract.groupby('addr_street')['addr_number'].count()
num_size = data_extract.groupby('addr_street').size()
num_value_counts = data_extract['addr_street'].value_counts()
diff = num_count - num_size
diff[diff != 0]

In [ ]:
assert num_size.sort_index().equals(num_value_counts.sort_index())

Exercise 3 grupby by 2 keys

In [ ]:
len_group = data_extract.groupby(['addr_street','condition']).size().sum()
print(len_group)

mask = data_extract['addr_street'].isna() | data_extract['source'].isna()
print(mask.sum())

In [ ]:
assert (mask.sum()+ len_group) == len(data_extract)

Exercise 4 aggregated functions per buildings per streets

In [ ]:
data_extract = data_extract.assign(num_func=lambda d: d['funkcja_parter'].str.count(',')+1) #adding another collumn based on the number of functions per building
data_extract.groupby('addr_street')['num_func'].agg(['size','min','max','mean']).head()

In [ ]:
data_extract.groupby('addr_street')['num_func'].get_group('Aleja Jana Pawła II').describe()

Exercise 5 Further aggregation

In [ ]:
data_extract.groupby('num_func', as_index=False).agg(
    n_buildings =('addr_street', 'size'),
    unique_streets = ('addr_street','nunique' ),
    n_fill_address = ('addr_number', 'count')
)

Exercise 6 Aggregation with custom function

In [ ]:
def max_min(arr):
    return arr.max() - arr.min()

cust_func_1 = data_extract.groupby('addr_street')['num_func'].agg(max_min)
cust_func_2 = data_extract.groupby('addr_street')['num_func'].agg(lambda s: s.max() - s.min())

In [ ]:
assert cust_func_1.equals(cust_func_2)

Exercise 7 transform

In [ ]:
data_extract = data_extract.assign(n_on_street = data_extract.groupby('addr_street')['osm_id'].transform('size'))
mask = data_extract['n_on_street'] >= 10
buildings_str_min_10 = data_extract[mask]

In [ ]:
assert len(data_extract) == 1105
assert buildings_str_min_10['n_on_street'].min() >= 10